In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

Cotporate Styling

In [ ]:
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
print("Everything is fine")

Load Dataset

In [ ]:
df=pd.read_csv("/content/sample_data/funnel_analysis_data.csv")
df.head()

Perform Basic Steps

In [ ]:
print("Top 5 data")
display(df.head())

In [ ]:
# Last 5 rows
df.tail()

In [ ]:
# Random data form dataset
df.sample()

In [ ]:
#find aTotal colummns name and number of columns
print(df.columns)
print(f"\nTotal Number of columns {df.columns.nunique()}")

In [ ]:
# type of dataset
type(df)

In [ ]:
df.dtypes

In [ ]:
df['Timestamp'] = pd.to_datetime('00:' + df['Timestamp'], format='%H:%M:%S.%f')
print(df.dtypes)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# imp info about all columns
df.describe(include="object")

In [ ]:
df.shape

In [ ]:
pd.set_option("display.max_rows", None)

Data Preprocessing And Cleaning

In [ ]:
#  check for the null and duplicate values
print("n\---Finding Null Values----\n")
null_values=df.isnull().sum()
print(null_values)

In [ ]:
print("n\---Finding Duplicate Values----\n")
duplicate_values=df.duplicated().sum()
print("Total numbers of duplicate_values in the dataset is {duplicate_values}")

In [ ]:
print("n\---Total Unique Data----")
unique_data=df.nunique()
print(unique_data)

In [ ]:
df['Event_sequence']=df.groupby('Session_ID').cumcount()+1

In [ ]:
df.head()

In [ ]:
# Basics steps
print(f'Total User ID{df['User_ID'].nunique()}')
print(f'\nTotal Session ID{df['Session_ID'].nunique()}')
print(f'\nTotal Events{df['Stage'].nunique()}')

Funnel Stage Definition and Session-Level Aggregation

In [ ]:
# Define funnel stages in order

funnel_stages=['Browse', 'Add to Cart', 'Checkout', 'Purchase']

# Create session-Level summary
session_summary = df.groupby('Session_ID').agg(
    User_ID=('User_ID', 'first'),
    Session_Start=('Timestamp', 'min'), # Capture the start time of the session
    Session_End=('Timestamp', 'max'),   # Capture the end time of the session
    Events=('Stage', lambda x: x.tolist()),
    Device=('Device', 'first'),
    Region=('Region', 'first'),
    Channel=('Channel', 'first'),
    # Product_Category=('Product_Category', 'first'), # This column does not exist in the original dataframe
    Revenue=('Revenue', 'sum'),
    Bounce_Flag=('Bounce_Flag', 'first')
).reset_index()

# The previous line to flatten column names is no longer needed as named aggregations define them directly.
# session_summary.columns=['Session_ID', 'User_ID', 'Timestamp', 'Events',
#                          'Device', 'Region', 'Channel',
#                          'Product_Category', 'Revenue', 'Bounce_Flag']

#Calculate session duration in minutes
session_summary['Session_Duration_Min']=(session_summary['Session_End']-session_summary['Session_Start']).dt.total_seconds()/60

In [ ]:
#identify max funnel stage reached for each session
def get_max_funnel_stage(events):
  stage_values={stage: i for i, stage in enumerate(funnel_stages)}
  max_stage_index=-1
  for event in events:
    # Ensure the event from the session_summary['Events'] list is in the defined funnel_stages
    if event in stage_values:
      max_stage_index=max(max_stage_index, stage_values[event])
  return funnel_stages[max_stage_index] if max_stage_index!= -1 else 'Browse'

# Apply the function to create the 'Max_Funnel_Stage' column
session_summary['Max_Funnel_Stage']=session_summary['Events'].apply(get_max_funnel_stage)
print("Session summary created")
display(session_summary.head())

Overall Funnel Analysis

In [ ]:
# Calculate overall funnel metrics
funnel_metrics = []
for i, stage in enumerate(funnel_stages):
    if i == 0:
        # For Browse stage, count all sessions
        count = len(session_summary)
    else:
        # For other stages,count sessions that reached at least this stage
        count = len(session_summary[session_summary['Max_Funnel_Stage'].isin(funnel_stages[i:])])

    funnel_metrics.append({
        'Stage': stage,
        'Sessions': count,
        'Stage_Order': i
    })

# Create DataFrame from funnel metrics
funnel_df = pd.DataFrame(funnel_metrics)

#  Calculate conversion and drop-off rates
funnel_df['Conversion_Rate'] = (funnel_df['Sessions'] / funnel_df['Sessions'].iloc[0] * 100).round(2)
funnel_df['Drop_Off_Rate'] = (1 - funnel_df['Sessions'] / funnel_df['Sessions'].shift(1)) * 100
funnel_df['Drop_Off_Rate'].iloc[0] = 0
funnel_df['Drop_Off_Rate'] = funnel_df['Drop_Off_Rate'].round(2)

print("Overall Funnel Analysis:")
display(funnel_df)

#Revenue analysis
revenue_stats = session_summary[session_summary['Max_Funnel_Stage'] == 'Purchase'].agg({
    'Revenue': ['sum', 'mean', 'count']
}).round(2)

print("\nRevenue Analysis:")
print(f"Total Revenue: ${revenue_stats.iloc[0, 0]:.2f}")
print(f"Average Order Value: ${revenue_stats.iloc[1, 0]:.2f}")
print(f"Total Orders: {revenue_stats.iloc[2, 0]}")


Visualization - Overall Funnel

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Create vis
fig=make_subplots(
    rows=2, cols=2,
    subplot_titles=('Funnel Conversion Rates', 'Stages to Stage Drop-off',
                    'Revenue by Funnel Stage', 'Session Duration by Stage'),
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"secondary_y": False}]]
)

#Funnel conversion rates
fig.add_trace(
    go.Bar(x=funnel_df['Stage'], y=funnel_df['Sessions'], # Changed 'Session' to 'Sessions'
           text=funnel_df['Sessions'], textposition='auto',
    name='Sessions', marker_color='lightblue'),
    row=1, col=1
)

# Drop-off rates
fig.add_trace(
    go.Scatter(
        x=funnel_df['Stage'],
        y=funnel_df['Drop_Off_Rate'],
        mode='lines+markers+text',
        text=funnel_df['Drop_Off_Rate'],
        textposition='top center',
        name='Drop-off Rate (%)',
        line=dict(color='red', width=3)
    ),
    row=1, col=2, secondary_y=False
)

# Revenue by stage
revenue_by_stage = session_summary.groupby('Max_Funnel_Stage')['Revenue'].sum().reset_index()
fig.add_trace(
    go.Bar(
        x=revenue_by_stage['Max_Funnel_Stage'],
        y=revenue_by_stage['Revenue'],
        text=[f'{x:.0f}' for x in revenue_by_stage['Revenue']],
        textposition='auto',
        name='Revenue',
        marker_color='green'
    ),
    row=2, col=1
)

# Session duration by stage
duration_by_stage = session_summary.groupby('Max_Funnel_Stage')['Session_Duration_Min'].mean().reset_index()
fig.add_trace(
    go.Bar(
        x=duration_by_stage['Max_Funnel_Stage'],
        y=duration_by_stage['Session_Duration_Min'],
        text=duration_by_stage['Session_Duration_Min'].round(2),
        textposition='auto',
        name='Avg Session Duration (min)',
        marker_color='orange'
    ),
    row=2, col=2 # Changed row from 3 to 2, and col to 2
)
fig.update_layout(height=800, title_text="Comprehensive Funnel Analysis Dashboard", showlegend=False)
fig.show()

Channel Performance Analysis

In [ ]:
# funnel analysis by channel
channel_funnel=[]
for channel in df['Channel'].unique():
  channel_sessions=session_summary[session_summary['Channel']==channel]
  total_sessions=len(channel_sessions)
  if total_sessions>0:
    channel_metrics={'Channel': channel, 'Total_Sessions': total_sessions}
    for i, stage in enumerate(funnel_stages):
      if i==0:
        count=total_sessions
      else:
        count=len(channel_sessions[channel_sessions['Max_Funnel_Stage'].isin(funnel_stages[i:])])
      channel_metrics[f'{stage}_Sessions']=count
      channel_metrics[f'{stage}_Rate']=(count/total_sessions*100)

    # Revenue metrics

    purchase_sessions=channel_sessions[channel_sessions['Max_Funnel_Stage']=='Purchase']
    channel_metrics['Total_Revenue']=purchase_sessions['Revenue'].sum()
    channel_metrics['Avg_Order_Value']=purchase_sessions['Revenue'].mean() if len(purchase_sessions)>0 else 0
    channel_metrics['Total_Orders']=(len(purchase_sessions)/total_sessions*100)
    channel_funnel.append(channel_metrics)
channel_df=pd.DataFrame(channel_funnel)
print("Performance Analysis: ")
display(channel_df.round(2))
#visualize channel performance
fig, ((ax1, ax2), (ax3, ax4)) =plt.subplots(2, 2, figsize=(15, 10))
# conversion rates by channel
sns.barplot(data=channel_df, x='Channel', y='Purchase_Rate', ax=ax1)
ax1.set_title('Conversion Rate by Channel')
ax1.tick_params(axis='x', rotation=45)
# Total revenue by  channel
sns.barplot(data=channel_df, x='Channel', y='Total_Revenue', ax=ax2)
ax2.set_title('Total Revenue by Channel')
ax2.tick_params(axis='x', rotation=45)

#AOV by channel
sns.barplot(data=channel_df, x='Channel', y='Avg_Order_Value', ax=ax3)
ax3.set_title('Average Order Value by Channel')
ax3.tick_params(axis='x', rotation=45)
#session distribution by channel
channel_df['Session_Percentage']=(channel_df['Total_Sessions']/channel_df['Total_Sessions'].sum()*100)
ax4.pie(channel_df['Session_Percentage'], labels=channel_df['Channel'], autopct='%1.1f%%')
ax4.set_title('Session Distribution by Channel')
plt.tight_layout()
plt.show()

Regional Analysis

In [ ]:
regional_analysis=session_summary.groupby('Region').agg(
    Total_Sessions=('Session_ID', 'count'),
    Total_Revenue=('Revenue', 'sum'),
    Avg_Session_Duration_Min=('Session_Duration_Min', 'mean')
)
# Add conversion rates by region
regional_conversion=session_summary[session_summary['Max_Funnel_Stage']=='Purchase'].groupby('Region').size()
regional_analysis['Converted_Sessions']=regional_conversion
regional_analysis['Conversion_Rate']=(regional_analysis['Converted_Sessions']/regional_analysis['Total_Sessions']*100).round(2)
regional_analysis['AOV']=(regional_analysis['Total_Revenue']/regional_analysis['Converted_Sessions']).round(2)
print("Regional Performance:")
display(regional_analysis)

#Regional funnel visualization
regional_funnel_data=[]
for region in df['Region'].unique():
  region_sessions=session_summary[session_summary['Region']==region]
  for stage in funnel_stages:
    if stage=='Browse':
      count=len(region_sessions)
    else:
      count=len(region_sessions[region_sessions['Max_Funnel_Stage'].isin(funnel_stages[funnel_stages.index(stage):])])
    regional_funnel_data.append({'Region': region, 'Stage': stage, 'Sessions': count})
regional_funnel_df=pd.DataFrame(regional_funnel_data)
plt.figure(figsize=(12, 6))
sns.barplot(data=regional_funnel_df, x='Region', y='Sessions', hue='Stage')
plt.title('Regional Funnel Analysis')
plt.xlabel('Region')
plt.ylabel('Sessions')
plt.show()

Device and Product  Category Analysis

In [ ]:
# Device performance
device_analysis = session_summary.groupby('Device').agg({
    'Session_ID': 'count',
    'Revenue': 'sum',
    'Session_Duration_Min': 'mean',
    'Max_Funnel_Stage': lambda x: (x == 'Purchase').sum()
}).rename(columns={'Session_ID': 'Total_Sessions', 'Max_Funnel_Stage': 'Purchases'})

device_analysis['Conversion_Rate'] = (device_analysis['Purchases'] / device_analysis['Total_Sessions'] * 100).round(2)
device_analysis['AOV'] = (device_analysis['Revenue'] / device_analysis['Purchases']).round(2)

print("# Device Performance")
display(device_analysis)

# Combined visualization
fig, ax1 = plt.subplots(1, 1, figsize=(8, 6))

#Device performance
sns.barplot(data=device_analysis.reset_index(), x='Device', y='Conversion_Rate', ax=ax1)
ax1.set_title('Conversion Rate by Device Type')

plt.tight_layout()
plt.show()

Time-Based Analysis

In [ ]:
#Time-Based Analysis

# Extract Date and Hour from Timestamp for grouping
df['Date'] = df['Timestamp'].dt.date
df['Hour'] = df['Timestamp'].dt.hour

# Daily Trends
daily_metrics = (
    df.groupby('Date')
      .agg({
          'Session_ID': 'nunique',
          'User_ID': 'nunique',
          'Revenue': 'sum'
    })
    .rename(columns={
          'Session_ID': 'Daily_Sessions',
          'User_ID': 'Daily_Users'
     })
)

#  Add Conversion Rates (Daily)
# Extract date from datetime for grouping
daily_conversions=(
    session_summary[session_summary['Max_Funnel_Stage'] == 'Purchase']
    .groupby(session_summary['Session_Start'].dt.date)
  .size()
)
daily_metrics['Daily_Conversions']=daily_conversions
daily_metrics['Daily_Conversion_Rate'] = (
    daily_metrics['Daily_Conversions']/daily_metrics['Daily_Sessions'] * 100
).round(2)

print("Daily Performance Trends")
display(daily_metrics.tail(10))

# Hourly Patterns
hourly_sessions = (
    df.groupby('Hour')
      .agg({
          'Session_ID': 'nunique',
          'Revenue': 'sum'
      })
      .rename(columns={'Session_ID': 'Hourly_Sessions'})
)

hourly_conversions =session_summary[
    session_summary['Max_Funnel_Stage'] == 'Purchase'
].copy()

hourly_conversions['Hour']= hourly_conversions['Session_Start'].dt.hour
hourly_conversion_counts= hourly_conversions.groupby('Hour').size()

hourly_sessions['Hourly_Conversions'] =hourly_conversion_counts

hourly_sessions['Hourly_Conversion_Rate'] = (
    hourly_sessions['Hourly_Conversions'] / hourly_sessions['Hourly_Sessions'] * 100
).round(2)


print(" Hourly Performance Trends")
display(hourly_sessions.tail(10))
# Visualization of Time-Based Metrics
import matplotlib.pyplot as plt

# Create a 2x2 grid of subplots
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

# Daily Sessions
ax1.plot(daily_metrics.index, daily_metrics['Daily_Sessions'], color='steelblue', linewidth=2)
ax1.set_title('Daily Sessions Over Time', fontsize=12, fontweight='bold')
ax1.set_xlabel('Date')
ax1.set_ylabel('Sessions')
ax1.tick_params(axis='x', rotation=45)

# --- Daily Conversion Rate ---
ax2.plot(daily_metrics.index, daily_metrics['Daily_Conversion_Rate'], color='darkorange', linewidth=2)
ax2.set_title('Daily Conversion Rate', fontsize=12, fontweight='bold')
ax2.set_xlabel('Date')
ax2.set_ylabel('Conversion Rate (%)')
ax2.tick_params(axis='x', rotation=45)

#Hourly Session Pattern
ax3.bar(hourly_sessions.index, hourly_sessions['Hourly_Sessions'], color='mediumseagreen')
ax3.set_title('Sessions by Hour of Day', fontsize=12, fontweight='bold')
ax3.set_xlabel('Hour of Day')
ax3.set_ylabel('Sessions')

#Hourly Conversion Rate
ax4.bar(hourly_sessions.index, hourly_sessions['Hourly_Conversion_Rate'], color='indianred')
ax4.set_title('Conversion Rate by Hour of Day', fontsize=12, fontweight='bold')
ax4.set_xlabel('Hour of Day')
ax4.set_ylabel('Conversion Rate (%)')

# Adjust layout for better spacing
plt.tight_layout()
plt.show()

Advance Funnel Metrics and KPIs

In [ ]:

print("🔸 KEY PERFORMANCE INDICATORS (KPIs)")
print("=" * 50)

#Overall KPIs
total_sessions = len(session_summary)
total_revenue = session_summary['Revenue'].sum()
total_orders = len(session_summary[session_summary['Max_Funnel_Stage'] == 'Purchase'])
overall_conversion_rate = (total_orders / total_sessions) * 100

print(f"🔥Overall Conversion Rate: {overall_conversion_rate:.2f}%")
print(f"💰Total Revenue: ${total_revenue:,.2f}")
print(f"📊Average Order Value: ${(total_revenue / total_orders):,.2f}")
print(f"🧮Total Sessions: {total_sessions:,}")
print(f"🛒Total Orders: {total_orders:,}")

# Funnel Efficiency Metrics
browse_to_cart = (funnel_df.iloc[1]['Sessions'] / funnel_df.iloc[0]['Sessions']) * 100
cart_to_checkout = (funnel_df.iloc[2]['Sessions'] / funnel_df.iloc[1]['Sessions']) * 100
checkout_to_purchase = (funnel_df.iloc[3]['Sessions'] / funnel_df.iloc[2]['Sessions']) * 100

print("\n📈Stage-to-Stage Conversion Rates:")
print(f"➡️Browse ➜ Add to Cart: {browse_to_cart:.2f}%")
print(f"➡️ Add to Cart ➜ Checkout: {cart_to_checkout:.2f}%")
print(f"➡️ Checkout ➜ Purchase: {checkout_to_purchase:.2f}%")

# Revenue per Session at Each Stage
revenue_per_browse = total_revenue / funnel_df.iloc[0]['Sessions']
revenue_per_cart = total_revenue / funnel_df.iloc[1]['Sessions']
revenue_per_checkout = total_revenue / funnel_df.iloc[2]['Sessions']

print("\n💵 Revenue per Session by Stage:")
print(f"🛍️Browse Stage: ${revenue_per_browse:,.2f}")
print(f"🛒 Add to Cart Stage: ${revenue_per_cart:,.2f}")
print(f"💳 Checkout Stage: ${revenue_per_checkout:,.2f}")
# Bounce Rate Analysis
bounce_sessions = session_summary[session_summary['Bounce_Flag'] == 'Yes']
bounce_rate = (len(bounce_sessions) / total_sessions) * 100
print(f"\n📉Bounce Rate: {bounce_rate:.2f}%")

#  SessionDurationAnalysis
avg_session_duration = session_summary['Session_Duration_Min'].mean()
print(f"⏱️ Average Session Duration: {avg_session_duration:.2f} minutes")

Strategic Recommendations

In [ ]:
# Funnel Performance Insights

# Identify Biggest Drop-Off Points
max_dropoff_stage = funnel_df.loc[funnel_df['Drop_Off_Rate'].idxmax()]
print(f"💥BIGGEST DROP-OFF: {max_dropoff_stage['Stage']} stage with {max_dropoff_stage['Drop_Off_Rate']:.2f}% drop-off")

# Best Performing Channel
best_channel = channel_df.loc[channel_df['Purchase_Rate'].idxmax()]
print(f"🏆BEST PERFORMING CHANNEL: {best_channel['Channel']} with {best_channel['Purchase_Rate']:.2f}% conversion")

# BestPerforming Region
best_region_name = "N/A (No conversions)"
best_region_conversion = 0.0
if regional_analysis['Converted_Sessions'].sum() > 0:
    best_region_data = regional_analysis.loc[regional_analysis['Conversion_Rate'].idxmax()]
    best_region_name = best_region_data.name
    best_region_conversion = best_region_data['Conversion_Rate']
print(f"🌍BEST PERFORMING REGION: {best_region_name} with {best_region_conversion:.2f}% conversion")


# Best Performing Product Category (Removed as 'Product_Category' column does not exist)
best_category_name = "N/A (No conversions)"
best_category_conversion = 0.0
# if product_analysis['Purchases'].sum() > 0:
#     best_category_data = product_analysis.loc[product_analysis['Conversion_Rate'].idxmax()]
#     best_category_name = best_category_data.name
#     best_category_conversion = best_category_data['Conversion_Rate']
print(f"🛍️ BEST PERFORMING CATEGORY: {best_category_name} with {best_category_conversion:.2f}% conversion")

# RecommendedActions
print("\n🔧 RECOMMENDED ACTIONS:")
print(f"1️⃣ Address {max_dropoff_stage['Stage']} stage drop-off through UX improvements.")
print(f"2️⃣ Allocate more budget to {best_channel['Channel']} channel.")
if best_region_name != "N/A (No conversions)":
    print(f"3️⃣ Replicate {best_region_name} region strategies in underperforming regions.")
else:
    print("3️⃣ Cannot make region-specific recommendations due to no conversions.")
# if best_category_name != "N/A (No conversions)": # Removed as 'Product_Category' column does not exist
#     print(f"4️⃣ Promote {best_category_name} category to improve overall conversion.")
# else:
print("4️⃣ Cannot make product category-specific recommendations due to no conversions.")
print(f"5️⃣ Focus on cart abandonment recovery for {funnel_df.iloc[2]['Drop_Off_Rate']:.2f}% of users.")

# Calculate PotentialRevenueOpportunities
# cart_abandonment_opportunity_value = funnel_df.iloc[2]['Sessions'] * product_analysis['AOV'].mean() # Removed as 'Product_Category' column does not exist
cart_abandonment_opportunity_value = 0 # Set to 0 or handle differently if AOV can be calculated from other metrics
if pd.isna(cart_abandonment_opportunity_value) or cart_abandonment_opportunity_value == 0:
    print("\n💰 REVENUE OPPORTUNITY: Not calculable due to no purchase data or zero potential revenue.")
else:
    print("\n💰 REVENUE OPPORTUNITY:")
    print(f"Cart abandonment recovery: ${cart_abandonment_opportunity_value:,.2f} potential revenue")

Result

In [ ]:
#Create Comprehensive Report
report_data = {
    'Overall_Funnel': funnel_df,
    'Channel_Performance': channel_df,
    'Regional_Analysis': regional_analysis,
    'Device_Performance': device_analysis,
    'Daily_Trends': daily_metrics,
    'Hourly_Patterns': hourly_sessions
}

# Export to Excel for Corporate Reporting
with pd.ExcelWriter('funnel_analysis_report.xlsx') as writer:
    for sheet_name, data in report_data.items():
        data.to_excel(writer, sheet_name=sheet_name, index=False)

print("Analysis complete! Report exported to 'funnel_analysis_report.xlsx'")

# Save Key Visualizations
plt.figure(figsize=(10, 6))
sns.barplot(data=funnel_df, x='Stage', y='Conversion_Rate', hue='Stage', palette='viridis', legend=False)
plt.title('Overall Funnel Conversion Rates', fontsize=14, fontweight='bold')
plt.xlabel('Stage')
plt.ylabel('Conversion Rate (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('funnel_conversion_rates.png', dpi=300, bbox_inches='tight')

print("📊 Key visualizations saved successfully!")